# TP 4 — RDD, lignage et DAG : prédire puis vérifier### Module 3 — Calcul distribué**Durée :** 2 heures · **Noté sur 20**---## Ce que vous devez savoir faire à la fin1. **Prédire** le nombre de jobs, de stages et de tâches d'un programme, **avant** de l'exécuter.2. Vérifier votre prédiction dans la Spark UI, et expliquer tout écart.3. Mesurer l'écart entre `groupByKey` et `reduceByKey` sur des données réelles.4. Observer l'effet du `cache()` sur un RDD réutilisé.5. Provoquer un déséquilibre et le reconnaître dans les métriques.## Méthode imposée> Pour chaque exercice marqué **PRÉDICTION**, vous devez écrire votre réponse> **avant d'exécuter la cellule**. Un exercice où la prédiction a été écrite après> coup ne vaut aucun point — et vous le sauriez.## Barème| Exercice | Sujet | Points ||---|---|---|| 1 | Mise en route et jeu de données | 2 || 2 | Prédire jobs et stages | 5 || 3 | `groupByKey` contre `reduceByKey` | 5 || 4 | L'effet du cache | 4 || 5 | Provoquer un déséquilibre | 4 |## LivrableCe notebook complété, exporté en HTML, déposé sur l'espace de cours.

---# Exercice 1 — Mise en route  *(2 points)***Objectif.** Disposer d'un jeu de données assez gros pour que les mesures aient un sens,et d'une session Spark dont vous connaissez les réglages.

In [ ]:
# 1.1 — Session Sparkfrom pyspark.sql import SparkSessionimport json, timeFILIERE = "if"        # "if" = Ingénierie Financière · "an" = Art NumériqueUTILISATEUR = "etudiant"spark = (SparkSession.builder         .appName("TP4 - RDD et DAG")         .master("local[4]")                    # 4 cœurs : assez pour observer         .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")         .config("spark.sql.shuffle.partitions", "8")         .getOrCreate())sc = spark.sparkContextsc.setLogLevel("WARN")print("Spark", spark.version, "| cœurs :", sc.defaultParallelism)print("Spark UI :", sc.uiWebUrl)

> **Gardez la Spark UI ouverte** dans un onglet pendant tout le TP : http://localhost:4040> Vous y reviendrez à chaque exercice.

In [ ]:
# 1.2 — Générer et déposer les données (à ne faire qu'une fois)!python /home/tinku/cours/99-Infra/scripts/generate_datasets.py \        --filiere {FILIERE} --sortie /home/tinku/work/data --evenements 2000000FICHIER = "if_transactions.jsonl" if FILIERE == "if" else "an_evenements.jsonl"!hdfs dfs -mkdir -p /user/{UTILISATEUR}/brut!hdfs dfs -put -f /home/tinku/work/data/{FICHIER} /user/{UTILISATEUR}/brut/!hdfs dfs -ls -h /user/{UTILISATEUR}/brut/{FICHIER}

In [ ]:
# 1.3 — Un RDD de départCHEMIN = f"hdfs://namenode:8020/user/{UTILISATEUR}/brut/{FICHIER}"lignes = sc.textFile(CHEMIN)print("Partitions du RDD source :", lignes.getNumPartitions())

### Q1 *(2 pts)* — Le nombre de partitions du RDD source correspond-il au nombre de blocsHDFS du fichier ? Vérifiez avec `hdfs fsck`, puis expliquez l'éventuel écart.

In [ ]:
!hdfs fsck /user/{UTILISATEUR}/brut/{FICHIER} | grep -E "Total blocks|Average block

**Votre réponse :***(rédigez ici)*

---# Exercice 2 — Prédire jobs et stages  *(5 points)***Objectif.** Vérifier que vous savez lire un programme Spark comme un DAG, et non commeune suite d'instructions.**Rappel.** stages = shuffles + 1 · tâches d'un stage = partitions à ce point · jobs = actions.

## 2.1 — **PRÉDICTION** *(2 pts)*Lisez le programme ci-dessous **sans l'exécuter**. Combien de **jobs**, combien de **stages** ?Où sont les frontières ? Écrivez votre réponse dans la cellule suivante, **puis** exécutez.```pythonpaires = (lignes          .map(json.loads)          .filter(lambda t: t.get(CHAMP_MONTANT) is not None)          .map(lambda t: (t[CHAMP_CLE], t[CHAMP_MONTANT])))totaux = paires.reduceByKey(lambda a, b: a + b)tries  = totaux.sortBy(lambda kv: -kv[1])print(tries.take(5))```

**Votre réponse :***(rédigez ici)*

In [ ]:
# 2.2 — Exécution. Notez le numéro du job dans la Spark UI.CHAMP_CLE     = "pays_transaction" if FILIERE == "if" else "pays"CHAMP_MONTANT = "montant"          if FILIERE == "if" else "position_s"paires = (lignes          .map(json.loads)          .filter(lambda t: t.get(CHAMP_MONTANT) is not None)          .map(lambda t: (t[CHAMP_CLE], t[CHAMP_MONTANT])))totaux = paires.reduceByKey(lambda a, b: a + b)tries  = totaux.sortBy(lambda kv: -kv[1])debut = time.time()print(tries.take(5))print(f"durée : {time.time() - debut:.1f} s")

In [ ]:
# 2.3 — Le lignage, tel que Spark le voitprint(tries.toDebugString().decode())

### Q2 *(3 pts)* — Dans la sortie de 2.3 :- **a.** Combien de `ShuffledRDD` comptez-vous ? Que confirme ce nombre ?- **b.** L'indentation change à chaque frontière de stage. Recopiez les trois groupes.- **c.** Ouvrez la Spark UI, onglet **Stages**. Le nombre de stages correspond-il à votre  prédiction ? Notez le volume *Shuffle Write* de chaque stage.

**Votre réponse :***(rédigez ici)*

---# Exercice 3 — `groupByKey` contre `reduceByKey`  *(5 points)***Objectif.** Mesurer, sur des données réelles, l'écart entre deux opérations qui produisentle même résultat.

## 3.1 — **PRÉDICTION** *(1 pt)*Les deux cellules qui suivent calculent **le même total par clé**. L'une utilise`reduceByKey`, l'autre `groupByKey` suivi d'un `sum`.Avant d'exécuter : laquelle sera la plus rapide, et **de quel ordre de grandeur** ?Surtout, **quelle métrique de la Spark UI** devrait montrer la différence ?

**Votre réponse :***(rédigez ici)*

In [ ]:
# 3.2 — reduceByKeysc.setJobDescription("A - reduceByKey")debut = time.time()r1 = paires.reduceByKey(lambda a, b: a + b).collect()t1 = time.time() - debutprint(f"reduceByKey : {t1:.1f} s — {len(r1)} clés distinctes")

In [ ]:
# 3.3 — groupByKeysc.setJobDescription("B - groupByKey")debut = time.time()r2 = paires.groupByKey().mapValues(sum).collect()t2 = time.time() - debutprint(f"groupByKey  : {t2:.1f} s — {len(r2)} clés distinctes")print(f"rapport     : {t2/t1:.1f}x")print("mêmes résultats :", sorted(r1) == sorted(r2))

### Q3 *(4 pts)* — Ouvrez la Spark UI, onglet **Stages**, et comparez les deux jobs.- **a.** Relevez le *Shuffle Write* de chacun. Quel rapport ?- **b.** Expliquez le mécanisme en une ou deux phrases.- **c.** Les deux produisent le même résultat. Existe-t-il un cas où `groupByKey` serait  néanmoins nécessaire ?- **d.** Quel **second** défaut `groupByKey` présente-t-il, indépendamment de la vitesse ?

**Votre réponse :***(rédigez ici)*

---# Exercice 4 — L'effet du cache  *(4 points)***Objectif.** Vérifier qu'un RDD non persisté est recalculé à chaque action.

## 4.1 — **PRÉDICTION** *(1 pt)*Le programme ci-dessous exécute **deux actions** sur le même RDD `base`, sans `cache()`.Combien de fois le fichier HDFS sera-t-il lu ? Justifiez.

**Votre réponse :***(rédigez ici)*

In [ ]:
# 4.2 — Sans cachebase = lignes.map(json.loads).filter(lambda t: t.get(CHAMP_MONTANT) is not None)sc.setJobDescription("C - sans cache")debut = time.time()n  = base.count()mx = base.map(lambda t: t[CHAMP_MONTANT]).max()sans_cache = time.time() - debutprint(f"sans cache : {sans_cache:.1f} s  ({n} lignes, max {mx})")

In [ ]:
# 4.3 — Avec cachebase_c = lignes.map(json.loads).filter(lambda t: t.get(CHAMP_MONTANT) is not None).cache()sc.setJobDescription("D - avec cache")base_c.count()                       # 1re action : remplit le cachedebut = time.time()n  = base_c.count()mx = base_c.map(lambda t: t[CHAMP_MONTANT]).max()avec_cache = time.time() - debutprint(f"avec cache : {avec_cache:.1f} s")print(f"rapport    : {sans_cache/avec_cache:.1f}x")

In [ ]:
# 4.4 — L'onglet Storage de la Spark UIprint("Ouvrez", sc.uiWebUrl + "/storage/")print("Taille en mémoire :", base_c.getStorageLevel())base_c.unpersist()          # toujours libérer ce dont on n'a plus besoin

### Q4 *(3 pts)* —- **a.** Quel gain avez-vous mesuré ? Est-il cohérent avec votre prédiction ?- **b.** Dans l'onglet **Storage**, quelle fraction du RDD est effectivement en mémoire ?- **c.** Donnez une règle simple pour décider quand mettre un RDD en cache — et un cas  où le faire serait une **erreur**.

**Votre réponse :***(rédigez ici)*

---# Exercice 5 — Provoquer un déséquilibre  *(4 points)***Objectif.** Reconnaître la signature d'un *skew* dans les métriques, avant de savoir lecorriger — ce sera l'objet du module 4.Les jeux de données du cours suivent volontairement une **loi de puissance** : quelquescomptes (ou quelques œuvres) concentrent une part disproportionnée de l'activité.

In [ ]:
# 5.1 — Observer la distribution des clésCLE_DESEQ = "id_compte" if FILIERE == "if" else "id_oeuvre"comptes = (lignes.map(json.loads)                 .map(lambda t: (t[CLE_DESEQ], 1))                 .reduceByKey(lambda a, b: a + b)                 .sortBy(lambda kv: -kv[1]))top = comptes.take(10)total = sum(c for _, c in comptes.collect())print("Top 10 des clés :")for k, c in top:    print(f"  {k}  {c:>8}  ({100*c/total:.2f} %)")print(f"\nLes 10 premières clés représentent {100*sum(c for _,c in top)/total:.1f} % du volume")

In [ ]:
# 5.2 — Un groupByKey sur cette clé déséquilibréesc.setJobDescription("E - groupByKey sur cle desequilibree")debut = time.time()res = (lignes.map(json.loads)             .map(lambda t: (t[CLE_DESEQ], t.get(CHAMP_MONTANT) or 0))             .groupByKey()             .mapValues(lambda v: sum(v))             .count())print(f"{res} clés en {time.time()-debut:.1f} s")

### Q5 *(4 pts)* — Ouvrez la Spark UI, onglet **Stages**, et cliquez sur le stage du shuffle.Faites défiler jusqu'au tableau **Summary Metrics for Tasks**.- **a.** Relevez la **durée médiane** et la **durée maximale** des tâches. Quel rapport ?- **b.** Relevez de même *Shuffle Read Size* médian et maximal.- **c.** Que se passerait-il si vous ajoutiez 100 machines au cluster ?- **d.** Sans chercher la solution complète, quelle **idée générale** permettrait de répartir  le travail d'une clé trop fréquente sur plusieurs tâches ?

**Votre réponse :***(rédigez ici)*

---# Synthèse| Ce que vous avez prédit | Vérifié ? | Écart éventuel et explication ||---|---|---|| Nombre de stages (ex. 2) | | || Le plus rapide entre `reduceByKey` et `groupByKey` (ex. 3) | | || Nombre de lectures du fichier sans cache (ex. 4) | | |**Une phrase de conclusion** : qu'est-ce qui, dans ce TP, vous a le plus surpris ?

In [ ]:
spark.stop()print("Session fermée.")

---## Avant de rendre- [ ] Les cinq **prédictions** sont écrites, et datées d'avant l'exécution.- [ ] Les questions **Q1 à Q5** sont rédigées et justifiées.- [ ] Les métriques relevées dans la Spark UI figurent dans vos réponses.- [ ] Le tableau de synthèse est complété.- [ ] Notebook exporté en HTML et déposé.**Bon TP.**